# 01 — El modelo sin RAG

Objetivo: observar qué puede responder el modelo sin acceder a las políticas ficticias de Andes Retail.
Este notebook no carga `corpus/` ni las respuestas esperadas, no usa búsqueda y no conserva una conversación entre consultas.

Ejecutar las celdas en orden con **Python (rag-demo)**. La celda de consulta realiza una llamada de API con costo por uso. Por defecto consulta solo Q01; repetir con Q09 es opcional.


## Cómo trabajar con este material

Un notebook combina **celdas Markdown**, que explican el razonamiento, con **celdas de código**, que ejecutan Python. Seleccioná una celda y presioná Shift+Enter para ejecutarla y avanzar. El kernel es el proceso de Python que mantiene las variables en memoria: cerrar una celda no borra esas variables.

Ejecutá de arriba hacia abajo. Si reiniciás el kernel, volvé a ejecutar desde el comienzo. Editar una celda no modifica automáticamente las variables: hay que ejecutarla de nuevo. Esto importa especialmente al cambiar una pregunta o una configuración.

Cada bloque tiene una explicación previa y una guía para interpretar su salida. Conviene hacer una pausa antes de ejecutar: **¿qué entra, qué transformación ocurre y qué debería salir?** No es necesario memorizar la sintaxis; sí poder relacionar cada operación con su propósito.


### Preparar la conexión: del archivo de configuración al cliente

Antes de preguntar al modelo necesitamos encontrar el proyecto y configurar una conexión. Todavía no estamos enviando una consulta.

| Elemento de Python | Qué hace y por qué aparece aquí |
|---|---|
| `from ... import ...` | Trae nombres de una biblioteca para usarlos en este notebook. `Path` administra rutas; `dotenv_values` lee la configuración; `OpenAI` construye el cliente. |
| `Path.cwd().resolve()` | Obtiene la carpeta de trabajo del kernel como ruta absoluta. Puede ser `rag-demo` o su subcarpeta `notebooks`. |
| `[carpeta, *carpeta.parents]` | Forma una lista con la carpeta actual y sus antecesoras. El asterisco incorpora los elementos de `parents` dentro de esa lista. |
| `next((p for p in ... if ...), None)` | Recorre esas carpetas y toma la primera que tiene las señales del proyecto: `requirements.in` y `corpus/`. Si no encuentra ninguna, devuelve `None`. |
| `raiz / ".env"` | Con `Path`, `/` une componentes de una ruta; aquí no es una división numérica. |
| `raise RuntimeError(...)` | Detiene la celda con una explicación si no se puede continuar. Es preferible a seguir con una configuración incorrecta. |

`dotenv_values` devuelve un **diccionario**: una colección de claves y valores. `config.get("OPENAI_API_KEY")` busca un valor sin provocar un error si la clave no existe. La codificación `utf-8-sig` admite archivos UTF-8 con o sin la marca inicial que algunos editores de Windows agregan.

La condición `not clave or not clave.strip()` detecta tanto un valor ausente como uno formado solo por espacios. `MODELO` usa la configuración si existe y, en caso contrario, el nombre predeterminado: esa es la función de `or` en esta asignación.

`OpenAI(...)` crea un **objeto cliente**, que reúne configuración y métodos para hablar con la API. `timeout=45.0` configura el tiempo de espera del cliente; `max_retries=0` desactiva reintentos automáticos para que un fallo sea visible en la demo. El cliente se conecta al servicio indicado por `base_url` cuando se hace una petición.

La última línea imprime solamente el modelo. `del clave` elimina ese nombre de variable, pero **no borra de forma segura la clave de la memoria**: la configuración y el cliente siguen necesitándola. No mostrar `config`, el contenido de `.env` ni las credenciales en las salidas.


In [ ]:
from pathlib import Path
from dotenv import dotenv_values
from openai import OpenAI, APIError, APIStatusError

carpeta = Path.cwd().resolve()
raiz = next((p for p in [carpeta, *carpeta.parents]
             if (p / "requirements.in").exists() and (p / "corpus").is_dir()), None)
if raiz is None:
    raise RuntimeError("Abrí el notebook dentro de rag-demo.")

config = dotenv_values(raiz / ".env", encoding="utf-8-sig")
clave = config.get("OPENAI_API_KEY")
if not clave or not clave.strip():
    raise RuntimeError("Falta OPENAI_API_KEY en rag-demo/.env.")

MODELO = config.get("OPENAI_MODEL") or "gpt-4.1-mini-2025-04-14"
cliente = OpenAI(api_key=clave, base_url="https://api.openai.com/v1",
                 timeout=45.0, max_retries=0)
del clave
print("Configuración cargada. Modelo:", MODELO)


**Cómo interpretar la salida.** Debe aparecer «Configuración cargada» y el nombre del modelo. Eso confirma que pudimos preparar el cliente; todavía no confirma que la clave sea aceptada ni que haya cuota disponible. La llamada posterior comprobará esas condiciones. Si la celda se detiene, corregí la ruta o la configuración antes de avanzar.

## Pregunta y contexto visibles

Usamos las preguntas Q01 y Q09 del set de evaluación, sin incluir sus soluciones. El contexto temporal es el mismo que utilizaremos al agregar RAG. El mensaje pide reconocer límites, sin insinuar la regla correcta.


### Construir la consulta antes de enviarla

Ahora armamos datos de entrada, todos visibles. `preguntas` es un diccionario que permite seleccionar el caso por su identificador: `preguntas[caso]` devuelve el texto asociado a `Q01` o `Q09`. Elegimos una sola pregunta para inspeccionar el comportamiento antes de automatizar una evaluación completa.

Separamos tres piezas por su función:

- **`instrucciones`**: cómo queremos que responda el asistente, incluido reconocer información insuficiente.
- **`contexto`**: empresa, fecha de consulta y supuestos compartidos del ejercicio. Este contexto no contiene las políticas ni sus reglas.
- **`entrada`**: ese contexto más la pregunta elegida. Es el contenido que recibirá la API como entrada del usuario.

Los paréntesis permiten escribir cadenas largas en varias líneas. Python concatena las cadenas literales contiguas; por eso mantenemos espacios al final de cada tramo. En `entrada`, el operador `+` une textos y `"\n\n"` agrega una línea en blanco entre contexto y pregunta.

¿Por qué imprimimos todo antes de llamar al modelo? Para observar exactamente qué información tiene disponible. El modelo no ve los archivos de nuestra computadora solo porque estén en la misma carpeta que el notebook. Tendríamos que leerlos y enviar su contenido explícitamente.


In [ ]:
preguntas = {
    "Q01": "¿Cuál es el plazo para solicitar reintegros de gastos de viaje pagados con dinero personal?",
    "Q09": "Compré con mi dinero un pasaje hace 75 días. ¿Todavía estoy a tiempo de solicitar el reintegro?",
}
caso = "Q01"  # Cambiar a Q09 y volver a ejecutar desde esta celda.
instrucciones = (
    "Respondé en español, de forma breve. Si no tenés información suficiente "
    "para responder, indicá qué falta. No inventes políticas ni fuentes."
)
contexto = (
    "Empresa: Andes Retail. Fecha de consulta: 2026-10-08. "
    "Salvo indicación contraria, los viajes finalizaron en 2026 y se pregunta "
    "por las reglas actuales. No inferir país de contratación, medio de pago "
    "ni fechas ausentes."
)
entrada = contexto + "\n\n" + preguntas[caso]
print(instrucciones)
print(entrada)


**Antes de continuar.** Leé el texto impreso: no debería aparecer ninguna regla de reintegros tomada del corpus. Cambiar `caso` y ejecutar esta celda actualiza `entrada`; para obtener una nueva respuesta también hay que ejecutar la siguiente celda. Esta preparación por sí sola no consume la API.

### Hacer la llamada y leer el resultado

Esta es la celda que utiliza el servicio externo. `cliente.responses.create(...)` envía una solicitud y devuelve un objeto de respuesta. El acceso con punto indica que usamos un método de `responses`, que pertenece al cliente creado antes.

| Argumento | Papel en el experimento |
|---|---|
| `model=MODELO` | Selecciona el modelo que resolverá la consulta. Lo mantendremos constante cuando comparemos con RAG. |
| `instructions=instrucciones` | Envía las pautas de comportamiento. |
| `input=entrada` | Envía la pregunta y el contexto que acabamos de inspeccionar. |
| `max_output_tokens=400` | Limita la longitud máxima de la salida; no significa 400 palabras ni obliga al modelo a usar todo el límite. |
| `store=False` | Desactiva el almacenamiento de la respuesta para recuperarla posteriormente mediante la API. No equivale a una garantía general de retención cero. |

No se pasan herramientas, documentos, identificadores de conversaciones ni respuestas previas. Cada ejecución hace una consulta independiente. Una respuesta distinta entre ejecuciones no implica necesariamente que el código haya cambiado.

`respuesta = None` evita reutilizar una respuesta anterior si la nueva llamada falla. El bloque `try` intenta ejecutar la petición. Si ocurre una excepción, Python pasa al `except` correspondiente:

- `APIStatusError` permite distinguir errores HTTP. El diccionario `mensajes` traduce algunos códigos en acciones concretas. Un 429 puede corresponder a cuota o a límites; este mensaje no diagnostica cuál de los dos.
- `APIError` cubre otros errores de la biblioteca, como problemas de conexión. Se coloca después del caso más específico para no ocultarlo.

Si la petición devolvió un objeto, `respuesta.output_text` muestra su texto. Los atributos `status`, `model` y `usage` permiten inspeccionar cómo terminó, qué modelo se usó y cuántos tokens se contabilizaron. Un token es una unidad de procesamiento de texto; no coincide necesariamente con una palabra.

`if respuesta is not None` evita acceder a atributos de una respuesta que nunca se recibió. El chequeo de `status` recuerda que una salida incompleta no debe evaluarse como si fuera una respuesta terminada. No imprimimos el objeto completo ni el cuerpo de los errores: solo la información útil para la clase.


In [ ]:
# Esta celda envía solamente las instrucciones y la entrada visibles arriba.
# Cada ejecución hace una nueva llamada; no se envían documentos ni herramientas.
respuesta = None
try:
    respuesta = cliente.responses.create(
        model=MODELO,
        instructions=instrucciones,
        input=entrada,
        max_output_tokens=400,
        store=False,
    )
except APIStatusError as error:
    mensajes = {
        401: "Revisá la clave en .env.",
        403: "Revisá los permisos del proyecto y del modelo.",
        404: "Revisá OPENAI_MODEL y el acceso al modelo.",
        429: "Revisá saldo/cuota y límites de solicitudes en la plataforma.",
    }
    print(f"API HTTP {error.status_code}.", mensajes.get(error.status_code, "Error del servicio; intentá más tarde."))
except APIError:
    print("No se completó la conexión. Revisá la red e intentá nuevamente.")

if respuesta is not None:
    print(respuesta.output_text)
    print("\nEstado:", respuesta.status)
    print("Modelo utilizado:", respuesta.model)
    if respuesta.usage:
        print("Tokens de entrada:", respuesta.usage.input_tokens)
        print("Tokens de salida:", respuesta.usage.output_tokens)
    if respuesta.status != "completed":
        print("La respuesta no está completa; no evaluarla como una respuesta terminada.")


**Cómo evaluar lo que apareció.** Separá dos preguntas: «¿la API funcionó?» y «¿la respuesta está sustentada?». Una solicitud exitosa solo responde la primera. Sin las políticas, una abstención es razonable y un plazo concreto necesita ser cuestionado. Anotá si el modelo reconoce el límite, inventa una regla o pide un dato. No hace falta repetir llamadas hasta lograr una equivocación: eso sesgaría la demostración.

## Discusión

1. ¿El modelo reconoce que desconoce la política interna o propone un plazo sin fuente?
2. ¿Qué información falta en Q09? Sin documentos, ¿puede siquiera conocer la regla aplicable?
3. ¿Qué tendría que aportar el retrieval para mejorar la respuesta?

Una abstención es un resultado válido: no buscamos forzar al modelo a equivocarse. Un plazo acertado por casualidad tampoco demuestra grounding. Las salidas pueden variar entre ejecuciones.

Conservar el modelo, las preguntas y las instrucciones al comparar con RAG; agregar luego únicamente el contexto recuperado. Revisar la rúbrica docente por separado después de obtener la respuesta.

Modelo inicial: snapshot de GPT-4.1 mini, elegido para una consulta breve sin razonamiento configurado. Puede cambiarse mediante `OPENAI_MODEL` en `.env`; mantener el mismo en ambas condiciones. `store=False` evita guardar la respuesta para recuperarla posteriormente mediante la API; no significa que no haya políticas de retención del proveedor.

Referencias: [SDK oficial](https://developers.openai.com/api/reference/python) · [modelo](https://developers.openai.com/api/docs/models/gpt-4.1-mini).
